# 03 — Causal Graphs and Structural Causal Models
**References:** Pearl (2009) *Causality* Ch. 1, 3 · Pearl, Glymour & Jewell (2016) *A Primer* Ch. 2–3 · Hernán & Robins (2020) Ch. 6 · UCLA CS 262 (Pearl)

## Narrative thread
```
Structural causal models -> DAGs -> The three elementary structures -> d-separation -> do-operator -> Backdoor & frontdoor criteria
```

## Structural causal models (SCMs)

A **structural causal model** is a set of assignments
$$X_j := f_j(\text{PA}_j,\ U_j)$$
where $\text{PA}_j$ are the direct causes (parents) of $X_j$ and $U_j$ is exogenous noise.
The corresponding **directed acyclic graph (DAG)** draws an arrow from each parent to its
child. The graph encodes *qualitative* causal knowledge; the $f_j$ carry the quantitative part.

## The three elementary structures

Every DAG is built from three patterns, each with different conditioning behavior:

| Structure | Graph | Marginal | Conditioning on $B$ |
|---|---|---|---|
| **Chain** (mediator) | $A \to B \to C$ | $A \not\perp C$ | $A \perp C \mid B$ — **blocks** |
| **Fork** (confounder) | $A \leftarrow B \to C$ | $A \not\perp C$ | $A \perp C \mid B$ — **blocks** |
| **Collider** | $A \to B \leftarrow C$ | $A \perp C$ | $A \not\perp C \mid B$ — **opens!** |

The collider is the counterintuitive one: $A$ and $C$ are independent, but conditioning on
their common *effect* creates dependence (Berkson's paradox).

## d-separation

A path is **blocked** by a conditioning set $Z$ if it contains a chain or fork whose middle
node is in $Z$, or a collider whose middle node (and all its descendants) is *not* in $Z$.
$X$ and $Y$ are **d-separated** by $Z$ if every path between them is blocked
$\Rightarrow X \perp Y \mid Z$ in every distribution compatible with the DAG.

## Intervention: the do-operator

$P(Y \mid do(X = x))$ is the distribution of $Y$ when $X$ is *set* to $x$ by intervention —
graphically, delete all arrows *into* $X$. Association $P(Y \mid X = x)$ conditions on
observation; intervention cuts the mechanisms that generate $X$. Confounding is precisely
$P(Y \mid do(x)) \neq P(Y \mid x)$.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── Draw the three elementary structures with matplotlib ─────────────────
def draw_dag(ax, nodes, edges, title, highlight=()):
    for name, (x, y) in nodes.items():
        color = '#ffd8a8' if name in highlight else '#d0e2ff'
        ax.add_patch(plt.Circle((x, y), 0.16, color=color, ec='k', zorder=3))
        ax.text(x, y, name, ha='center', va='center', zorder=4, fontsize=11)
    for a, b in edges:
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        dx, dy = x2 - x1, y2 - y1
        d = np.hypot(dx, dy)
        sx, sy = x1 + 0.19*dx/d, y1 + 0.19*dy/d
        ex, ey = x2 - 0.19*dx/d, y2 - 0.19*dy/d
        ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                    arrowprops=dict(arrowstyle='-|>', lw=1.6, color='k'))
    ax.set_xlim(-0.4, 2.4); ax.set_ylim(-0.5, 1.4)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(11, 2.9))
draw_dag(axes[0], {'A': (0, 0), 'B': (1, 1), 'C': (2, 0)},
         [('A', 'B'), ('B', 'C')], 'Chain: A -> B -> C', highlight='B')
draw_dag(axes[1], {'A': (0, 0), 'B': (1, 1), 'C': (2, 0)},
         [('B', 'A'), ('B', 'C')], 'Fork: A <- B -> C  (confounder)', highlight='B')
draw_dag(axes[2], {'A': (0, 1), 'B': (1, 0), 'C': (2, 1)},
         [('A', 'B'), ('C', 'B')], 'Collider: A -> B <- C', highlight='B')
plt.tight_layout(); plt.show()

In [ ]:
# ── Verify the conditioning rules by simulation ──────────────────────────
n = 100_000
def corr_given(a, c, b, cond):
    """Correlation of a, c within a slice of b (or marginal if cond is None)."""
    if cond is None:
        return np.corrcoef(a, c)[0, 1]
    m = np.abs(b - cond) < 0.25          # condition on a thin slice of B
    return np.corrcoef(a[m], c[m])[0, 1]

# Chain: A -> B -> C
a = np.random.normal(0, 1, n); b = 1.5*a + np.random.normal(0, 1, n)
c = 1.5*b + np.random.normal(0, 1, n)
print(f'Chain    : corr(A,C) = {corr_given(a,c,b,None):+.3f}   corr(A,C | B~0) = {corr_given(a,c,b,0):+.3f}')

# Fork: A <- B -> C
b = np.random.normal(0, 1, n)
a = 1.5*b + np.random.normal(0, 1, n); c = 1.5*b + np.random.normal(0, 1, n)
print(f'Fork     : corr(A,C) = {corr_given(a,c,b,None):+.3f}   corr(A,C | B~0) = {corr_given(a,c,b,0):+.3f}')

# Collider: A -> B <- C
a = np.random.normal(0, 1, n); c = np.random.normal(0, 1, n)
b = a + c + np.random.normal(0, 0.5, n)
print(f'Collider : corr(A,C) = {corr_given(a,c,b,None):+.3f}   corr(A,C | B~0) = {corr_given(a,c,b,0):+.3f}')
print('\nConditioning BLOCKS chains and forks, but OPENS colliders.')

## The backdoor criterion

A set $Z$ satisfies the **backdoor criterion** relative to $(X, Y)$ if:
1. no node in $Z$ is a descendant of $X$, and
2. $Z$ blocks every path from $X$ to $Y$ that starts with an arrow **into** $X$.

Then the causal effect is identified by the **adjustment formula**:
$$P(Y \mid do(x)) = \sum_z P(Y \mid x, z)\, P(z)$$

For means, this is standardization / g-formula:
$E[Y \mid do(x)] = E_Z\big[E[Y \mid X = x, Z]\big]$ — the engine behind regression
adjustment, matching, and propensity-score methods (notebooks 05–06).

## The frontdoor criterion

When the confounder is **unobserved**, identification can still succeed through a
mediator $M$ with $X \to M \to Y$ if (i) $M$ intercepts all directed paths $X \to Y$,
(ii) no unblocked backdoor from $X$ to $M$, (iii) all backdoors from $M$ to $Y$ are
blocked by $X$:
$$P(y \mid do(x)) = \sum_m P(m \mid x) \sum_{x'} P(y \mid m, x')\, P(x')$$

Pearl's classic example: smoking ($X$) → tar deposits ($M$) → cancer ($Y$), with an
unobserved genotype confounding $X$ and $Y$.


In [ ]:
# ── Backdoor adjustment recovers the causal effect ───────────────────────
# DAG:  Z -> X, Z -> Y, X -> Y      (true effect of X on Y is 2.0)
n = 300_000
z = np.random.normal(0, 1, n)
x = 1.2*z + np.random.normal(0, 1, n)
y = 2.0*x + 3.0*z + np.random.normal(0, 1, n)
df = pd.DataFrame({'x': x, 'y': y, 'z': z})

naive    = smf.ols('y ~ x', df).fit().params['x']
adjusted = smf.ols('y ~ x + z', df).fit().params['x']
print(f'True causal effect:            2.000')
print(f'Naive regression  y ~ x:       {naive:.3f}   (confounded)')
print(f'Backdoor adjusted y ~ x + z:   {adjusted:.3f}   (identified)')

In [ ]:
# ── Frontdoor identification with an UNOBSERVED confounder ───────────────
# DAG:  U -> X, U -> Y  (U unobserved),  X -> M -> Y
# Binary version so we can apply the frontdoor formula directly.
n = 2_000_000
u = np.random.binomial(1, 0.5, n)
x = np.random.binomial(1, 0.2 + 0.6*u)              # U -> X
m = np.random.binomial(1, 0.1 + 0.7*x)              # X -> M
y = np.random.binomial(1, np.clip(0.1 + 0.5*m + 0.3*u, 0, 1))   # M -> Y <- U

# Ground truth by simulation of do(X=x): cut U -> X
def truth(xv):
    mm = np.random.binomial(1, 0.1 + 0.7*xv, n)
    return np.mean(np.random.binomial(1, np.clip(0.1 + 0.5*mm + 0.3*u, 0, 1)))
true_effect = truth(1) - truth(0)

# Frontdoor formula: sum_m P(m|x) sum_x' P(y|m,x') P(x')
p_x = np.array([np.mean(x == 0), np.mean(x == 1)])
def p_m_given_x(mv, xv): return np.mean(m[x == xv] == mv)
def p_y_given_mx(mv, xv): return np.mean(y[(m == mv) & (x == xv)])

def frontdoor(xv):
    total = 0.0
    for mv in (0, 1):
        inner = sum(p_y_given_mx(mv, xp) * p_x[xp] for xp in (0, 1))
        total += p_m_given_x(mv, xv) * inner
    return total

fd = frontdoor(1) - frontdoor(0)
naive = y[x == 1].mean() - y[x == 0].mean()
print(f'True effect  E[Y|do(X=1)] - E[Y|do(X=0)]: {true_effect:.4f}')
print(f'Naive difference (confounded by U):        {naive:.4f}')
print(f'Frontdoor estimate (U never observed):     {fd:.4f}')

## do-calculus (the general machinery)

Pearl's three rules transform expressions with $do(\cdot)$ into do-free (estimable)
expressions when identification is possible; backdoor and frontdoor are special cases.
Completeness (Shpitser & Pearl 2006): if do-calculus cannot remove the $do$, the effect is
not identifiable from that graph.

## Two languages, one subject

| | Potential outcomes (Rubin) | Graphs (Pearl) |
|---|---|---|
| Primitive | $Y_i(w)$ | SCM / DAG |
| Unconfoundedness | $W \perp (Y(0),Y(1)) \mid X$ | $X$ satisfies backdoor criterion |
| Strength | Estimation theory, uncertainty | Transparent assumptions, identification algorithms |

They are logically equivalent (an SCM implies potential outcomes as $Y(w) = f_Y(w, U)$);
use DAGs to *choose* the adjustment set, potential outcomes to *estimate* the effect.

## Key takeaways

| Concept | Statement |
|---|---|
| Chain / fork / collider | Conditioning blocks the first two, opens the third |
| d-separation | Graphical test for conditional independence |
| $do(x)$ | Intervention = delete arrows into $X$; not the same as conditioning |
| Backdoor criterion | Valid adjustment set ⟹ $E[Y\mid do(x)] = E_Z[E[Y\mid x,Z]]$ |
| Frontdoor criterion | Identification through a clean mediator despite unobserved confounding |
| Equivalence | Rubin's and Pearl's frameworks translate into each other |
